# 08 머신러닝 예측 모델 (Phase 1 — 2/3)

**12조건**(SBC 8 + ML 4, 2-type E·C) × **RF, XGBoost** | 지표: MAE, RMSE, MAPE, MASE

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 66
학습 <= 201707 | 검증: [201708, 201709, 201710, 201711, 201712, 201713, 201714, 201715, 201716, 201717, 201718, 201719, 201720]
Best 선정 기준: MAPE


### ① ML 모델 실험

In [2]:
# Phase 1 — ML 2종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_ml_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=ML_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=ML_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_ml_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_ml_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/8 [00:00<?, ?it/s]

Phase1 ML:   0%|          | 0/8 [00:00<?, ?it/s]

완료 | 264 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
1,ML,C,1,XGBoost,971.641542,1793.216626,212.040395,1.856109
3,ML,C,2,XGBoost,42936.740362,59379.341613,35.691973,2.888354
5,ML,E,1,XGBoost,781.491004,1173.690364,100.074887,2.787332
7,ML,E,2,XGBoost,14612.161896,22322.276167,23.696725,1.945015
8,SBC,C,1,RF,5957.324111,9637.527075,39.861309,1.847918
10,SBC,C,2,RF,942.623335,1721.203401,46.211349,1.933771
12,SBC,C,3,RF,625.274922,1345.984460,33.180611,0.394615
15,SBC,C,4,XGBoost,12.069904,15.731958,281.141853,0.553491
17,SBC,E,1,XGBoost,2343.321424,3558.395771,48.695289,1.936537
19,SBC,E,2,XGBoost,470.094209,898.038375,47.768925,2.212609



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML   SBC
model                       
RF              199.90  59.3
XGBoost         146.12  54.1
